# Main Rerun Single-Stage Ablation


# One-Stage Daily Raw Rich Feature Engineering


# Main Rerun One-Stage Raw-Rich Feature Runtime Config

- `STAGE_SPLIT_SCHEME = "main_rerun_primary_holdout"`
- 正式 `primary holdout = 2025-07-15 ~ 2025-10-25`

In [1]:
from pathlib import Path
import os

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "plan_v2.md").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)

MAIN_RERUN_STAGE2_DIR = PROJECT_ROOT / "main_rerun" / "artifacts" / "stage2"
MAIN_RERUN_ABLATION_DIR = PROJECT_ROOT / "main_rerun" / "artifacts" / "ablation" / "one_stage_daily_raw_rich"
MAIN_RERUN_ABLATION_DIR.mkdir(parents=True, exist_ok=True)

os.environ["STAGE_SPLIT_SCHEME"] = "main_rerun_primary_holdout"
os.environ["STAGE2_HOLDOUT_LABEL"] = "primary_holdout"
os.environ["STAGE2_BASE_PATH"] = str(
    MAIN_RERUN_STAGE2_DIR / "stage2_daily_features_wide_v2_main_rerun.csv"
)
os.environ["ONE_STAGE_ARTIFACTS_DIR"] = str(MAIN_RERUN_ABLATION_DIR)

print("Project root:", PROJECT_ROOT)
print("Main rerun Stage 2 base dir:", MAIN_RERUN_STAGE2_DIR)
print("Main rerun ablation output dir:", MAIN_RERUN_ABLATION_DIR)

Project root: /Users/horace/Coding/ML finance/project
Main rerun Stage 2 base dir: /Users/horace/Coding/ML finance/project/main_rerun/artifacts/stage2
Main rerun ablation output dir: /Users/horace/Coding/ML finance/project/main_rerun/artifacts/ablation/one_stage_daily_raw_rich


In [2]:
from pathlib import Path
import os

os.environ["MPLCONFIGDIR"] = str(Path.cwd() / ".mpl-cache")

import warnings

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="talk")

ROOT = Path.cwd()
if ROOT.name == "ablation":
    ROOT = ROOT.parent

ARTIFACTS_DIR = Path(os.environ.get("ONE_STAGE_ARTIFACTS_DIR", ROOT / "ablation" / "artifacts"))
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

EVENT_PATH = ROOT / "data" / "integration" / "event_stage1_aligned_features_v1.csv"
STAGE2_BASE_PATH = Path(
    os.environ.get(
        "STAGE2_BASE_PATH",
        ROOT / "stage2" / "artifacts" / "stage2_daily_features_wide_v2.csv",
    )
)

BRIDGE_OUTPUT_PATH = ARTIFACTS_DIR / "one_stage_daily_raw_rich_event_daily_bridge_v1.csv"
DAILY_OUTPUT_PATH = ARTIFACTS_DIR / "one_stage_daily_raw_rich_features_v1.csv"
QC_OUTPUT_PATH = ARTIFACTS_DIR / "one_stage_daily_raw_rich_feature_qc_summary_v1.csv"

NEG_THRESHOLD = 0.50
POS_THRESHOLD = 0.50

CORE_CONTROL_FEATURES = [
    "dxy_ret_1d_lag1",
    "real_yield_change_5d_lag1",
    "vix_ma_5d_lag1",
    "vix_change_5d_lag1",
    "spx_ret_1d_lag1",
    "spx_drawdown_5d_lag1",
    "gold_spx_corr_20d_lag1",
]

TREND_FEATURES = [
    "trend_tariff_z_60d_lag1",
    "trend_inflation_z_60d_lag1",
    "trend_war_z_60d_lag1",
    "trend_tariff_spike_lag1",
    "trend_inflation_spike_lag1",
    "trend_war_spike_lag1",
]

FORBIDDEN_FEATURE_COLS = [
    "label_retreat",
    "p_taco",
    "follow_up_count_all_48h",
    "follow_up_count_relevant_48h",
    "theme_hits",
    "action_hits",
    "object_hits",
]

In [3]:
SPLIT_SCHEME = os.environ.get("STAGE_SPLIT_SCHEME", "primary_holdout")

TRAIN_LABEL = "train_pool"
HOLDOUT_LABEL = os.environ.get("STAGE2_HOLDOUT_LABEL", "primary_holdout")
OUTSIDE_LABEL = "outside_main_sample"

SPLIT_SCHEMES = {
    "primary_holdout": {
        "train_ranges": [
            ("2017-01-20", "2021-01-08"),
            ("2025-01-20", "2025-07-31"),
        ],
        "holdout_range": ("2025-08-01", "2025-10-25"),
    },
    "main_rerun_primary_holdout": {
        "train_ranges": [
            ("2017-01-20", "2021-01-08"),
            ("2025-01-20", "2025-07-14"),
        ],
        "holdout_range": ("2025-07-15", "2025-10-25"),
    },
    "first_term_final_year_holdout": {
        "train_ranges": [
            ("2017-01-20", "2019-06-30"),
        ],
        "holdout_range": ("2019-07-01", "2020-02-28"),
    },
}


def describe_split_scheme(split_scheme):
    scheme = SPLIT_SCHEMES[split_scheme]
    return {
        "split_scheme": split_scheme,
        "train_ranges": scheme["train_ranges"],
        "holdout_range": scheme["holdout_range"],
        "train_label": TRAIN_LABEL,
        "holdout_label": HOLDOUT_LABEL,
    }


def classify_daily_split(date_values, split_scheme=SPLIT_SCHEME):
    scheme = SPLIT_SCHEMES[split_scheme]
    dates = pd.to_datetime(pd.Series(date_values)).dt.normalize()
    tz = getattr(dates.dt, "tz", None)
    if tz is not None:
        dates = dates.dt.tz_convert(None)

    train_mask = pd.Series(False, index=dates.index)
    for start_text, end_text in scheme["train_ranges"]:
        start = pd.Timestamp(start_text)
        end = pd.Timestamp(end_text)
        train_mask |= (dates >= start) & (dates <= end)

    holdout_start, holdout_end = scheme["holdout_range"]
    holdout_mask = (dates >= pd.Timestamp(holdout_start)) & (dates <= pd.Timestamp(holdout_end))
    return np.select(
        [train_mask, holdout_mask],
        [TRAIN_LABEL, HOLDOUT_LABEL],
        default=OUTSIDE_LABEL,
    )


print("Split scheme:", describe_split_scheme(SPLIT_SCHEME))

Split scheme: {'split_scheme': 'main_rerun_primary_holdout', 'train_ranges': [('2017-01-20', '2021-01-08'), ('2025-01-20', '2025-07-14')], 'holdout_range': ('2025-07-15', '2025-10-25'), 'train_label': 'train_pool', 'holdout_label': 'primary_holdout'}


## Load And Normalize Inputs


In [4]:
def classify_stage2_date(date_series):
    return classify_daily_split(
        date_series,
        split_scheme=SPLIT_SCHEME,
    )


def load_event_source():
    df = pd.read_csv(EVENT_PATH)

    df["event_time_utc"] = pd.to_datetime(df["event_time_utc"], format="mixed", utc=True)
    df["feature_anchor_date"] = pd.to_datetime(df["feature_anchor_date"], format="mixed", utc=True)
    df["target_trade_date"] = pd.to_datetime(df["target_trade_date"], format="mixed", utc=True)
    df["target_trade_date_naive"] = df["target_trade_date"].dt.tz_convert(None).dt.normalize()
    df["event_date_ny"] = df["event_time_utc"].dt.tz_convert("America/New_York").dt.normalize()

    bool_defaults = {
        "is_trump_direct_text": False,
        "is_retweet": False,
        "in_market_hours": False,
    }
    for col, default in bool_defaults.items():
        if col in df.columns:
            df[col] = df[col].fillna(default).astype(bool)
        else:
            df[col] = default

    numeric_defaults = {
        "keyword_score": 0.0,
        "finbert_pos": 0.0,
        "finbert_neu": 0.0,
        "finbert_neg": 0.0,
    }
    for col, default in numeric_defaults.items():
        if col not in df.columns:
            raise ValueError(f"Missing required event column: {col}")
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(default)

    return df


def load_stage2_daily_base():
    df = pd.read_csv(STAGE2_BASE_PATH)
    df["date"] = pd.to_datetime(df["date"]).dt.normalize()

    expected_split = classify_stage2_date(df["date"])
    mismatch_count = int((df["stage2_split"] != expected_split).sum())
    print("stage2 split mismatches vs protocol:", mismatch_count)

    keep_cols = [
        "date",
        "stage2_split",
        "is_main_stage2_sample",
        "gold_close",
        "gold_ret_1d",
    ] + CORE_CONTROL_FEATURES + TREND_FEATURES

    missing_keep_cols = [col for col in keep_cols if col not in df.columns]
    if missing_keep_cols:
        raise ValueError(f"Missing stage2 base columns: {missing_keep_cols}")

    base = (
        df[keep_cols]
        .drop_duplicates(subset=["date"])
        .sort_values("date")
        .reset_index(drop=True)
    )
    return base, mismatch_count

In [5]:
event_df = load_event_source()
daily_base, split_mismatch_count = load_stage2_daily_base()

print("event_df rows:", event_df.shape)
print("daily_base rows:", daily_base.shape)
display(event_df[[
    "event_id",
    "event_time_utc",
    "target_trade_date_naive",
    "finbert_pos",
    "finbert_neu",
    "finbert_neg",
    "candidate_priority" if "candidate_priority" in event_df.columns else "source",
]].head())
display(daily_base.head())

stage2 split mismatches vs protocol: 0
event_df rows: (1886, 86)
daily_base rows: (2293, 18)


,event_id,event_time_utc,target_trade_date_naive,finbert_pos,finbert_neu,finbert_neg,candidate_priority
0,FIRST_TERM_TWITTER_824440456813707265,2017-01-26 02:14:56+00:00,2017-01-26,0.037903,0.917480,0.044617,low
1,FIRST_TERM_TWITTER_824615820391305216,2017-01-26 13:51:46+00:00,2017-01-26,0.048340,0.094849,0.856934,low
2,FIRST_TERM_TWITTER_824616644370714627,2017-01-26 13:55:03+00:00,2017-01-26,0.007835,0.033997,0.958008,high
3,FIRST_TERM_TWITTER_824970003153842176,2017-01-27 13:19:10+00:00,2017-01-27,0.063843,0.144287,0.791992,high
4,FIRST_TERM_TWITTER_825017279209410561,2017-01-27 16:27:02+00:00,2017-01-27,0.314453,0.672363,0.013039,low


,date,stage2_split,is_main_stage2_sample,gold_close,gold_ret_1d,dxy_ret_1d_lag1,real_yield_change_5d_lag1,vix_ma_5d_lag1,vix_change_5d_lag1,spx_ret_1d_lag1,spx_drawdown_5d_lag1,gold_spx_corr_20d_lag1,trend_tariff_z_60d_lag1,trend_inflation_z_60d_lag1,trend_war_z_60d_lag1,trend_tariff_spike_lag1,trend_inflation_spike_lag1,trend_war_spike_lag1
0,2016-11-01,outside_main_sample,0,1302.519715,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2016-11-02,outside_main_sample,0,1308.589123,0.004649,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0
2,2016-11-03,outside_main_sample,0,1311.193116,0.001988,-0.000622,NaN,NaN,NaN,-0.006547,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0
3,2016-11-04,outside_main_sample,0,1311.596841,0.000308,-0.001354,NaN,19.986667,NaN,-0.004433,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0
4,2016-11-07,outside_main_sample,0,1299.637957,-0.009160,-0.001615,NaN,20.617500,NaN,-0.001668,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0


## Build Event-Level Helper Columns

这一步只构造 richer raw aggregation 所需的 event-level helper columns，
不直接产出 modeling table。

In [6]:
def add_event_level_helper_columns(df):
    out = df.copy()

    if "candidate_priority" in out.columns:
        priority = out["candidate_priority"].astype(str).str.strip().str.lower()
        out["candidate_priority_high_flag"] = priority.eq("high").astype(int)
    else:
        out["candidate_priority_high_flag"] = 0

    out["keyword_score_filled"] = pd.to_numeric(out.get("keyword_score", 0.0), errors="coerce").fillna(0.0)

    source_norm = out.get("source", pd.Series("", index=out.index)).astype(str).str.strip().str.lower()
    out["source_twitter_flag"] = source_norm.eq("twitter").astype(int)
    out["source_truthsocial_flag"] = source_norm.eq("truthsocial").astype(int)

    term_norm = out.get("term_id", pd.Series("", index=out.index)).astype(str).str.strip().str.lower()
    out["term_first_flag"] = term_norm.eq("first_term").astype(int)
    out["term_second_flag"] = term_norm.eq("second_term").astype(int)

    out["finbert_pos"] = out["finbert_pos"].clip(0.0, 1.0)
    out["finbert_neu"] = out["finbert_neu"].clip(0.0, 1.0)
    out["finbert_neg"] = out["finbert_neg"].clip(0.0, 1.0)

    out["extreme_neg_flag"] = (out["finbert_neg"] >= NEG_THRESHOLD).astype(int)
    out["extreme_pos_flag"] = (out["finbert_pos"] >= POS_THRESHOLD).astype(int)

    return out


event_df_enriched = add_event_level_helper_columns(event_df)

helper_cols = [
    "candidate_priority_high_flag",
    "keyword_score_filled",
    "source_twitter_flag",
    "source_truthsocial_flag",
    "term_first_flag",
    "term_second_flag",
    "extreme_neg_flag",
    "extreme_pos_flag",
]
display(event_df_enriched[["event_id"] + helper_cols].head())

,event_id,candidate_priority_high_flag,keyword_score_filled,source_twitter_flag,source_truthsocial_flag,term_first_flag,term_second_flag,extreme_neg_flag,extreme_pos_flag
0,FIRST_TERM_TWITTER_824440456813707265,0,2,1,0,1,0,0,0
1,FIRST_TERM_TWITTER_824615820391305216,0,6,1,0,1,0,1,0
2,FIRST_TERM_TWITTER_824616644370714627,1,3,1,0,1,0,1,0
3,FIRST_TERM_TWITTER_824970003153842176,1,6,1,0,1,0,1,0
4,FIRST_TERM_TWITTER_825017279209410561,0,2,1,0,1,0,0,0


## Aggregate Rich Daily Raw Features

richer single-stage 的核心是：

- 保留 `target_trade_date` 级别聚合
- 但不再只做窄口径的 negative sentiment
- 而是把 richer FinBERT / event structure / priority / source / term 组都带下来

In [7]:
def aggregate_daily_raw_rich_features(events):
    grouped_rows = []

    for target_date, group in events.groupby("target_trade_date_naive"):
        pos = group["finbert_pos"].astype(float)
        neu = group["finbert_neu"].astype(float)
        neg = group["finbert_neg"].astype(float)

        grouped_rows.append(
            {
                "date": target_date,
                "finbert_pos_max": float(pos.max()),
                "finbert_pos_mean": float(pos.mean()),
                "finbert_neg_max": float(neg.max()),
                "finbert_neg_mean": float(neg.mean()),
                "finbert_neu_mean": float(neu.mean()),
                "finbert_neg_top1": float(neg.max()),
                "finbert_neg_top2_sum": float(neg.nlargest(min(2, len(neg))).sum()),
                "finbert_neg_any": float(1.0 - np.prod(1.0 - neg.to_numpy())),
                "event_count": int(len(group)),
                "direct_text_count": int(group["is_trump_direct_text"].sum()),
                "retweet_count": int(group["is_retweet"].sum()),
                "market_hours_event_count": int(group["in_market_hours"].sum()),
                "extreme_neg_count": int(group["extreme_neg_flag"].sum()),
                "extreme_pos_count": int(group["extreme_pos_flag"].sum()),
                "keyword_score_max": float(group["keyword_score_filled"].max()),
                "keyword_score_mean": float(group["keyword_score_filled"].mean()),
                "candidate_priority_high_count": int(group["candidate_priority_high_flag"].sum()),
                "source_twitter_count": int(group["source_twitter_flag"].sum()),
                "source_truthsocial_count": int(group["source_truthsocial_flag"].sum()),
                "term_first_count": int(group["term_first_flag"].sum()),
                "term_second_count": int(group["term_second_flag"].sum()),
                "min_event_time_utc": group["event_time_utc"].min(),
                "max_event_time_utc": group["event_time_utc"].max(),
            }
        )

    aggregated = pd.DataFrame(grouped_rows).sort_values("date").reset_index(drop=True)
    return aggregated


def add_temporal_text_features(daily):
    ordered = daily.sort_values("date").copy()
    signal_cols = [
        "finbert_neg_any",
        "finbert_neg_top1",
        "finbert_neg_top2_sum",
    ]
    for col in signal_cols:
        ordered[f"{col}_decay_2d"] = (
            ordered[col]
            + 0.5 * ordered[col].shift(1).fillna(0.0)
        )
        ordered[f"{col}_decay_3d"] = (
            ordered[col]
            + 0.5 * ordered[col].shift(1).fillna(0.0)
            + 0.25 * ordered[col].shift(2).fillna(0.0)
        )
    return ordered


aggregated_daily_raw = aggregate_daily_raw_rich_features(event_df_enriched)
aggregated_daily_raw = add_temporal_text_features(aggregated_daily_raw)

print("aggregated_daily_raw rows:", aggregated_daily_raw.shape)
display(aggregated_daily_raw.head())

aggregated_daily_raw rows: (741, 30)


,date,finbert_pos_max,finbert_pos_mean,finbert_neg_max,finbert_neg_mean,finbert_neu_mean,finbert_neg_top1,finbert_neg_top2_sum,finbert_neg_any,event_count,...,term_first_count,term_second_count,min_event_time_utc,max_event_time_utc,finbert_neg_any_decay_2d,finbert_neg_any_decay_3d,finbert_neg_top1_decay_2d,finbert_neg_top1_decay_3d,finbert_neg_top2_sum_decay_2d,finbert_neg_top2_sum_decay_3d
0,2017-01-26,0.048340,0.031359,0.958008,0.619853,0.348775,0.958008,1.814941,0.994260,3,...,3,0,2017-01-26 02:14:56+00:00,2017-01-26 13:55:03+00:00,0.994260,0.994260,0.958008,0.958008,1.814941,1.814941
1,2017-01-27,0.314453,0.189148,0.791992,0.402515,0.408325,0.791992,0.805031,0.794704,2,...,2,0,2017-01-27 13:19:10+00:00,2017-01-27 16:27:02+00:00,1.291835,1.291835,1.270996,1.270996,1.712502,1.712502
2,2017-02-01,0.051056,0.051056,0.027267,0.027267,0.921875,0.027267,0.027267,0.027267,1,...,1,0,2017-02-01 00:31:08+00:00,2017-02-01 00:31:08+00:00,0.424620,0.673185,0.423264,0.662766,0.429783,0.883518
3,2017-02-02,0.143555,0.134827,0.583984,0.304604,0.560547,0.583984,0.609207,0.594477,2,...,2,0,2017-02-02 11:34:36+00:00,2017-02-02 11:39:49+00:00,0.608111,0.806787,0.597618,0.795616,0.622841,0.824099
4,2017-02-06,0.040558,0.040558,0.068481,0.068481,0.891113,0.068481,0.068481,0.068481,1,...,1,0,2017-02-04 03:07:47+00:00,2017-02-04 03:07:47+00:00,0.365720,0.372537,0.360474,0.367290,0.373085,0.379902


## Merge With Daily Base

这一步固定使用 `date` 把 richer daily raw 表 merge 回统一的 daily base，
保持与当前 two-stage 相同的 split / target / controls。

In [8]:
def build_daily_raw_rich_table(daily_base, aggregated_daily_raw):
    merged = daily_base.merge(
        aggregated_daily_raw,
        on="date",
        how="left",
        validate="one_to_one",
    )

    fill_zero_cols = [
        "finbert_pos_max",
        "finbert_pos_mean",
        "finbert_neg_max",
        "finbert_neg_mean",
        "finbert_neu_mean",
        "finbert_neg_top1",
        "finbert_neg_top2_sum",
        "finbert_neg_any",
        "event_count",
        "direct_text_count",
        "retweet_count",
        "market_hours_event_count",
        "extreme_neg_count",
        "extreme_pos_count",
        "keyword_score_max",
        "keyword_score_mean",
        "candidate_priority_high_count",
        "source_twitter_count",
        "source_truthsocial_count",
        "term_first_count",
        "term_second_count",
        "finbert_neg_any_decay_2d",
        "finbert_neg_any_decay_3d",
        "finbert_neg_top1_decay_2d",
        "finbert_neg_top1_decay_3d",
        "finbert_neg_top2_sum_decay_2d",
        "finbert_neg_top2_sum_decay_3d",
    ]
    for col in fill_zero_cols:
        merged[col] = merged[col].fillna(0.0)

    int_cols = [
        "event_count",
        "direct_text_count",
        "retweet_count",
        "market_hours_event_count",
        "extreme_neg_count",
        "extreme_pos_count",
        "candidate_priority_high_count",
        "source_twitter_count",
        "source_truthsocial_count",
        "term_first_count",
        "term_second_count",
    ]
    for col in int_cols:
        merged[col] = merged[col].astype(int)

    merged["has_one_stage_signal"] = (merged["event_count"] > 0).astype(int)
    return merged


daily_raw_rich = build_daily_raw_rich_table(daily_base, aggregated_daily_raw)
print("daily_raw_rich rows:", daily_raw_rich.shape)
display(daily_raw_rich.head())

daily_raw_rich rows: (2293, 48)


,date,stage2_split,is_main_stage2_sample,gold_close,gold_ret_1d,dxy_ret_1d_lag1,real_yield_change_5d_lag1,vix_ma_5d_lag1,vix_change_5d_lag1,spx_ret_1d_lag1,...,term_second_count,min_event_time_utc,max_event_time_utc,finbert_neg_any_decay_2d,finbert_neg_any_decay_3d,finbert_neg_top1_decay_2d,finbert_neg_top1_decay_3d,finbert_neg_top2_sum_decay_2d,finbert_neg_top2_sum_decay_3d,has_one_stage_signal
0,2016-11-01,outside_main_sample,0,1302.519715,NaN,NaN,NaN,NaN,NaN,NaN,...,0,NaT,NaT,0.0,0.0,0.0,0.0,0.0,0.0,0
1,2016-11-02,outside_main_sample,0,1308.589123,0.004649,NaN,NaN,NaN,NaN,NaN,...,0,NaT,NaT,0.0,0.0,0.0,0.0,0.0,0.0,0
2,2016-11-03,outside_main_sample,0,1311.193116,0.001988,-0.000622,NaN,NaN,NaN,-0.006547,...,0,NaT,NaT,0.0,0.0,0.0,0.0,0.0,0.0,0
3,2016-11-04,outside_main_sample,0,1311.596841,0.000308,-0.001354,NaN,19.986667,NaN,-0.004433,...,0,NaT,NaT,0.0,0.0,0.0,0.0,0.0,0.0,0
4,2016-11-07,outside_main_sample,0,1299.637957,-0.009160,-0.001615,NaN,20.617500,NaN,-0.001668,...,0,NaT,NaT,0.0,0.0,0.0,0.0,0.0,0.0,0


## Quality Checks

- richer 表与 protocol split 是否一致
- richer 表是否只新增 raw groups，而没有混入 `P_taco` 或 label-window 派生列
- richer raw feature 缺失情况
- 多事件聚合分布

In [9]:
forbidden_present = [col for col in FORBIDDEN_FEATURE_COLS if col in daily_raw_rich.columns]
if forbidden_present:
    raise ValueError(f"Forbidden columns present in richer daily table: {forbidden_present}")

event_per_target = (
    event_df_enriched.groupby("target_trade_date_naive")
    .size()
    .rename("n_events")
    .reset_index()
)

qc_rows = [
    {"metric": "event_rows", "value": float(len(event_df_enriched))},
    {"metric": "daily_base_rows", "value": float(len(daily_base))},
    {"metric": "daily_raw_rich_rows", "value": float(len(daily_raw_rich))},
    {"metric": "stage2_split_mismatch_count", "value": float(split_mismatch_count)},
    {"metric": "signal_days", "value": float(daily_raw_rich["has_one_stage_signal"].sum())},
    {"metric": "signal_day_share", "value": float(daily_raw_rich["has_one_stage_signal"].mean())},
    {"metric": "target_dates_with_multi_events", "value": float((event_per_target["n_events"] > 1).sum())},
    {"metric": "target_dates_with_multi_events_share", "value": float((event_per_target["n_events"] > 1).mean())},
    {"metric": "max_events_same_target_trade_date", "value": float(event_per_target["n_events"].max())},
    {"metric": "mean_events_per_target_trade_date", "value": float(event_per_target["n_events"].mean())},
]

missing_feature_cols = [
    "finbert_pos_max",
    "finbert_neg_any",
    "event_count",
    "keyword_score_max",
    "candidate_priority_high_count",
    "source_twitter_count",
    "term_first_count",
] + CORE_CONTROL_FEATURES + TREND_FEATURES

for col in missing_feature_cols:
    qc_rows.append(
        {
            "metric": f"missing_rate::{col}",
            "value": float(daily_raw_rich[col].isna().mean()),
        }
    )

qc_summary = pd.DataFrame(qc_rows)

display(
    daily_raw_rich.groupby("stage2_split")
    .agg(
        n_days=("date", "count"),
        n_signal_days=("has_one_stage_signal", "sum"),
        avg_event_count=("event_count", "mean"),
        avg_keyword_score_max=("keyword_score_max", "mean"),
        avg_finbert_neg_any=("finbert_neg_any", "mean"),
    )
)
display(
    event_per_target["n_events"]
    .describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99])
    .to_frame("value")
)
display(qc_summary.head(20))

,n_days,n_signal_days,avg_event_count,avg_keyword_score_max,avg_finbert_neg_any
stage2_split,,,,,
outside_main_sample,1089,0,0.000000,0.000000,0.000000
primary_holdout,74,53,1.689189,4.472973,0.265999
train_pool,1130,687,1.557522,3.076991,0.248669


,value
count,741.000000
mean,2.545209
std,2.285009
min,1.000000
50%,2.000000
75%,3.000000
90%,5.000000
95%,7.000000
99%,11.600000
max,23.000000


,metric,value
0,event_rows,1886.000000
1,daily_base_rows,2293.000000
2,daily_raw_rich_rows,2293.000000
3,stage2_split_mismatch_count,0.000000
4,signal_days,740.000000
5,signal_day_share,0.322721
6,target_dates_with_multi_events,454.000000
7,target_dates_with_multi_events_share,0.612686
8,max_events_same_target_trade_date,23.000000
9,mean_events_per_target_trade_date,2.545209


In [10]:
plot_df = daily_raw_rich[daily_raw_rich["is_main_stage2_sample"] == 1].copy()

fig, axes = plt.subplots(4, 1, figsize=(16, 16), sharex=True)

sns.lineplot(data=plot_df, x="date", y="finbert_neg_any", ax=axes[0], linewidth=1.8)
axes[0].set_title("Rich single-stage daily finbert_neg_any")
axes[0].set_xlabel("")
axes[0].set_ylabel("neg_any")

sns.lineplot(data=plot_df, x="date", y="event_count", ax=axes[1], linewidth=1.6)
axes[1].set_title("Rich single-stage daily event_count")
axes[1].set_xlabel("")
axes[1].set_ylabel("event_count")

sns.lineplot(data=plot_df, x="date", y="keyword_score_max", ax=axes[2], linewidth=1.6)
axes[2].set_title("Rich single-stage daily keyword_score_max")
axes[2].set_xlabel("")
axes[2].set_ylabel("keyword_score_max")

sns.lineplot(data=plot_df, x="date", y="candidate_priority_high_count", ax=axes[3], linewidth=1.4)
axes[3].set_title("Rich single-stage daily candidate_priority_high_count")
axes[3].set_xlabel("date")
axes[3].set_ylabel("high_priority_count")

plt.tight_layout()
plt.show()

## Export

- event-to-day bridge
- richer daily raw feature 表
- QC summary

In [11]:
bridge_cols = [
    "event_id",
    "text_id",
    "source",
    "term_id",
    "event_time_utc",
    "event_date_ny",
    "feature_anchor_date",
    "target_trade_date",
    "target_trade_date_naive",
    "finbert_pos",
    "finbert_neu",
    "finbert_neg",
    "keyword_score_filled",
    "candidate_priority_high_flag",
    "is_trump_direct_text",
    "is_retweet",
    "in_market_hours",
    "source_twitter_flag",
    "source_truthsocial_flag",
    "term_first_flag",
    "term_second_flag",
]
bridge_export = event_df_enriched[bridge_cols].copy()

keep_daily_cols = [
    "date",
    "stage2_split",
    "is_main_stage2_sample",
    "gold_close",
    "gold_ret_1d",
    "finbert_pos_max",
    "finbert_pos_mean",
    "finbert_neg_max",
    "finbert_neg_mean",
    "finbert_neu_mean",
    "finbert_neg_top1",
    "finbert_neg_top2_sum",
    "finbert_neg_any",
    "finbert_neg_any_decay_2d",
    "finbert_neg_any_decay_3d",
    "finbert_neg_top1_decay_2d",
    "finbert_neg_top1_decay_3d",
    "finbert_neg_top2_sum_decay_2d",
    "finbert_neg_top2_sum_decay_3d",
    "event_count",
    "direct_text_count",
    "retweet_count",
    "market_hours_event_count",
    "extreme_neg_count",
    "extreme_pos_count",
    "has_one_stage_signal",
    "keyword_score_max",
    "keyword_score_mean",
    "candidate_priority_high_count",
    "source_twitter_count",
    "source_truthsocial_count",
    "term_first_count",
    "term_second_count",
] + CORE_CONTROL_FEATURES + TREND_FEATURES + [
    "min_event_time_utc",
    "max_event_time_utc",
]
daily_export = daily_raw_rich[keep_daily_cols].copy()

forbidden_in_export = [col for col in FORBIDDEN_FEATURE_COLS if col in daily_export.columns]
if forbidden_in_export:
    raise ValueError(f"Forbidden columns present in export: {forbidden_in_export}")

bridge_export.to_csv(BRIDGE_OUTPUT_PATH, index=False)
daily_export.to_csv(DAILY_OUTPUT_PATH, index=False)
qc_summary.to_csv(QC_OUTPUT_PATH, index=False)

print("Saved:")
print(" -", BRIDGE_OUTPUT_PATH)
print(" -", DAILY_OUTPUT_PATH)
print(" -", QC_OUTPUT_PATH)
print()
print("Daily richer feature columns:")
print(daily_export.columns.tolist())

Saved:
 - /Users/horace/Coding/ML finance/project/main_rerun/artifacts/ablation/one_stage_daily_raw_rich/one_stage_daily_raw_rich_event_daily_bridge_v1.csv
 - /Users/horace/Coding/ML finance/project/main_rerun/artifacts/ablation/one_stage_daily_raw_rich/one_stage_daily_raw_rich_features_v1.csv
 - /Users/horace/Coding/ML finance/project/main_rerun/artifacts/ablation/one_stage_daily_raw_rich/one_stage_daily_raw_rich_feature_qc_summary_v1.csv

Daily richer feature columns:
['date', 'stage2_split', 'is_main_stage2_sample', 'gold_close', 'gold_ret_1d', 'finbert_pos_max', 'finbert_pos_mean', 'finbert_neg_max', 'finbert_neg_mean', 'finbert_neu_mean', 'finbert_neg_top1', 'finbert_neg_top2_sum', 'finbert_neg_any', 'finbert_neg_any_decay_2d', 'finbert_neg_any_decay_3d', 'finbert_neg_top1_decay_2d', 'finbert_neg_top1_decay_3d', 'finbert_neg_top2_sum_decay_2d', 'finbert_neg_top2_sum_decay_3d', 'event_count', 'direct_text_count', 'retweet_count', 'market_hours_event_count', 'extreme_neg_count', 'ex

# One-Stage Daily Raw Rich Modeling

- 在固定 `RandomForestRegressor`、固定 protocol split、固定 core controls 的前提下
- richer `daily raw representation`
- 能否缩小 `single-stage` 与当前 `two-stage final` 的差距


- `single_stage_daily_raw_rich`
- `RandomForestRegressor`
- `linear_svm__sigmoid + RandomForest Regressor + two_stage_v2_core`

## Inputs And Outputs

输入：

- `ablation/artifacts/one_stage_daily_raw_rich_features_v1.csv`
- `main_rerun/artifacts/stage2/final_compare/`

输出：

- `ablation/artifacts/one_stage_daily_raw_rich_modeling_protocol_v1/outer_splits.csv`
- `best_params_by_fold.csv`
- `outer_fold_metrics.csv`
- `outer_summary_metrics.csv`
- `holdout_metrics.csv`
- `holdout_predictions.csv`
- `holdout_importances.csv`
- `refreshed_one_stage_vs_two_stage_table.csv`
- `refreshed_one_stage_vs_two_stage_summary.md`

# Main Rerun One-Stage Raw-Rich Modeling Runtime Config

- holdout benchmark ：
  - `Random Walk (Zero)`
  - `Ridge + macro_only`
  - `RandomForest + macro_only`

In [ ]:
from pathlib import Path
import os

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "plan_v2.md").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)

MAIN_RERUN_STAGE2_FINAL_DIR = PROJECT_ROOT / "main_rerun" / "artifacts" / "stage2" / "final_compare"
MAIN_RERUN_ABLATION_FEATURE_DIR = PROJECT_ROOT / "main_rerun" / "artifacts" / "ablation" / "one_stage_daily_raw_rich"
MAIN_RERUN_ABLATION_MODELING_DIR = PROJECT_ROOT / "main_rerun" / "artifacts" / "ablation" / "one_stage_daily_raw_rich_modeling"
MAIN_RERUN_ABLATION_MODELING_DIR.mkdir(parents=True, exist_ok=True)

os.environ["STAGE2_HOLDOUT_LABEL"] = "primary_holdout"
os.environ["STAGE2_HOLDOUT_DISPLAY_TITLE"] = "Primary Holdout"
os.environ["ONE_STAGE_RICH_DATA_PATH"] = str(
    MAIN_RERUN_ABLATION_FEATURE_DIR / "one_stage_daily_raw_rich_features_v1.csv"
)
os.environ["ONE_STAGE_RICH_MODELING_RESULTS_DIR"] = str(MAIN_RERUN_ABLATION_MODELING_DIR)
os.environ["ONE_STAGE_TWO_STAGE_HOLDOUT_PATH"] = str(
    MAIN_RERUN_STAGE2_FINAL_DIR / "stage2_nested_cv_model_compare_holdout_metrics.csv"
)
os.environ["ONE_STAGE_TWO_STAGE_OUTER_SUMMARY_PATH"] = str(
    MAIN_RERUN_STAGE2_FINAL_DIR / "stage2_nested_cv_model_compare_selected_model.csv"
)
os.environ["ONE_STAGE_BENCHMARK_HOLDOUT_PATH"] = str(
    MAIN_RERUN_STAGE2_FINAL_DIR / "stage2_nested_cv_model_compare_holdout_benchmark_metrics.csv"
)

print("Project root:", PROJECT_ROOT)
print("Main rerun ablation feature dir:", MAIN_RERUN_ABLATION_FEATURE_DIR)
print("Main rerun ablation modeling dir:", MAIN_RERUN_ABLATION_MODELING_DIR)
print("Two-stage holdout anchor:", os.environ["ONE_STAGE_TWO_STAGE_HOLDOUT_PATH"])
print("Two-stage outer anchor:", os.environ["ONE_STAGE_TWO_STAGE_OUTER_SUMMARY_PATH"])
print("Benchmark holdout file:", os.environ["ONE_STAGE_BENCHMARK_HOLDOUT_PATH"])

In [ ]:
from pathlib import Path
import json
import os

os.environ["MPLCONFIGDIR"] = str(Path.cwd() / ".mpl-cache")

import warnings

import matplotlib
matplotlib.use("Agg")
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "plan_v2.md").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "ablation" / "artifacts" / "one_stage_daily_raw_rich_features_v1.csv"

RESULTS_DIR = PROJECT_ROOT / "ablation" / "artifacts" / "one_stage_daily_raw_rich_modeling_protocol_v1"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

TARGET_COL = "gold_ret_1d"
DATE_COL = "date"
MODEL_NAME = "random_forest_regressor"
SPEC_NAME = "single_stage_daily_raw_rich"
TRAIN_LABEL = "train_pool"
HOLDOUT_LABEL = os.environ.get("STAGE2_HOLDOUT_LABEL", "primary_holdout")
HOLDOUT_DISPLAY_TITLE = os.environ.get("STAGE2_HOLDOUT_DISPLAY_TITLE", "Primary Hold-out")

OUTER_N_SPLITS = 5
OUTER_PURGE_DAYS = 2
OUTER_EMBARGO_DAYS = 5
OUTER_MIN_TRAIN_DAYS = 180
OUTER_MIN_VALID_DAYS = 30

INNER_N_SPLITS = 3
INNER_PURGE_DAYS = 2
INNER_EMBARGO_DAYS = 2
INNER_MIN_TRAIN_DAYS = 120
INNER_MIN_VALID_DAYS = 20

GAP_SPLIT_DAYS = 30
MIN_PREFIX_DAYS_LATER_SEGMENT = 30
RANDOM_STATE = 42


DATA_PATH = Path(os.environ.get("ONE_STAGE_RICH_DATA_PATH", str(DATA_PATH)))
RESULTS_DIR = Path(os.environ.get("ONE_STAGE_RICH_MODELING_RESULTS_DIR", str(RESULTS_DIR)))
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
benchmark_holdout_path_text = os.environ.get("ONE_STAGE_BENCHMARK_HOLDOUT_PATH", "").strip()
BENCHMARK_HOLDOUT_PATH = Path(benchmark_holdout_path_text) if benchmark_holdout_path_text else None


def resolve_two_stage_reference_inputs():
    direct_reference_path = os.environ.get("ONE_STAGE_TWO_STAGE_REFERENCE_PATH")
    holdout_reference_path = os.environ.get("ONE_STAGE_TWO_STAGE_HOLDOUT_PATH")
    outer_reference_path = os.environ.get("ONE_STAGE_TWO_STAGE_OUTER_SUMMARY_PATH")

    if holdout_reference_path and outer_reference_path:
        return (
            None,
            Path(holdout_reference_path),
            Path(outer_reference_path),
        )

    if direct_reference_path:
        return Path(direct_reference_path), None, None

    candidates = [
        PROJECT_ROOT / "stage2" / "final" / "stage2_selected_candidate_metrics.csv",
        PROJECT_ROOT / "stage2" / "final" / "stage2_all_model_results_table.csv",
        PROJECT_ROOT / "stage2" / "final" / "old" / "stage2_selected_candidate_metrics.csv",
    ]
    for path in candidates:
        if path.exists():
            return path, None, None
    raise FileNotFoundError("Could not locate a two-stage final reference file under stage2/final/.")


TWO_STAGE_REFERENCE_PATH, TWO_STAGE_REFERENCE_HOLDOUT_PATH, TWO_STAGE_REFERENCE_OUTER_SUMMARY_PATH = (
    resolve_two_stage_reference_inputs()
)
if TWO_STAGE_REFERENCE_PATH is not None:
    print("Using two-stage reference:", TWO_STAGE_REFERENCE_PATH.relative_to(PROJECT_ROOT))
else:
    print(
        "Using two-stage holdout/outer references:",
        TWO_STAGE_REFERENCE_HOLDOUT_PATH.relative_to(PROJECT_ROOT),
        TWO_STAGE_REFERENCE_OUTER_SUMMARY_PATH.relative_to(PROJECT_ROOT),
    )
if BENCHMARK_HOLDOUT_PATH is not None:
    print("Using rerun benchmark reference:", BENCHMARK_HOLDOUT_PATH.relative_to(PROJECT_ROOT))


In [3]:
CORE_CONTROL_FEATURES = [
    "dxy_ret_1d_lag1",
    "real_yield_change_5d_lag1",
    "vix_ma_5d_lag1",
    "vix_change_5d_lag1",
    "spx_ret_1d_lag1",
    "spx_drawdown_5d_lag1",
    "gold_spx_corr_20d_lag1",
]

RICH_FINBERT_FEATURES = [
    "finbert_pos_max",
    "finbert_pos_mean",
    "finbert_neg_max",
    "finbert_neg_mean",
    "finbert_neu_mean",
    "finbert_neg_any",
    "finbert_neg_top1",
    "finbert_neg_top2_sum",
    "finbert_neg_any_decay_2d",
    "finbert_neg_any_decay_3d",
    "finbert_neg_top1_decay_2d",
    "finbert_neg_top1_decay_3d",
    "finbert_neg_top2_sum_decay_2d",
    "finbert_neg_top2_sum_decay_3d",
]

EVENT_STRUCTURE_FEATURES = [
    "event_count",
    "direct_text_count",
    "retweet_count",
    "market_hours_event_count",
    "extreme_neg_count",
    "extreme_pos_count",
    "has_one_stage_signal",
    "keyword_score_max",
    "keyword_score_mean",
    "candidate_priority_high_count",
    "source_twitter_count",
    "source_truthsocial_count",
    "term_first_count",
    "term_second_count",
]

TREND_FEATURES = [
    "trend_tariff_z_60d_lag1",
    "trend_inflation_z_60d_lag1",
    "trend_war_z_60d_lag1",
    "trend_tariff_spike_lag1",
    "trend_inflation_spike_lag1",
    "trend_war_spike_lag1",
]

MODEL_FEATURES = (
    CORE_CONTROL_FEATURES
    + RICH_FINBERT_FEATURES
    + EVENT_STRUCTURE_FEATURES
    + TREND_FEATURES
)


def compute_regression_metrics(y_true, y_pred):
    return {
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "r2": float(r2_score(y_true, y_pred)),
        "sign_accuracy": float((np.sign(y_pred) == np.sign(y_true)).mean()),
        "actual_mean": float(np.mean(y_true)),
        "pred_mean": float(np.mean(y_pred)),
    }


def split_day_segments(unique_days, gap_days=30):
    unique_days = pd.Index(sorted(pd.Index(unique_days).unique()))
    if len(unique_days) == 0:
        return []

    segments = []
    start = 0
    diffs = np.diff(unique_days.view("int64")) / (24 * 3600 * 1e9)
    for idx, gap in enumerate(diffs, start=1):
        if gap > gap_days:
            segments.append(unique_days[start:idx])
            start = idx
    segments.append(unique_days[start:])
    return segments


def allocate_splits_to_segments(
    segments,
    n_splits,
    min_train_days,
    purge_days,
    min_valid_days,
    embargo_days,
    min_prefix_days_later_segment,
):
    eligible = []
    for seg_idx, segment in enumerate(segments):
        required_days = (
            min_train_days + purge_days + min_valid_days
            if seg_idx == 0
            else min_prefix_days_later_segment + purge_days + min_valid_days
        )
        if len(segment) < required_days:
            continue

        segment_capacity = int(
            (len(segment) - required_days) // max(min_valid_days + embargo_days, 1)
        ) + 1
        if segment_capacity <= 0:
            continue

        eligible.append(
            {
                "segment_idx": seg_idx,
                "segment_len": len(segment),
                "capacity": segment_capacity,
            }
        )

    if not eligible:
        raise ValueError("No eligible day segments available for the requested split design.")

    allocation = {item["segment_idx"]: 0 for item in eligible}
    remaining = n_splits

    for item in eligible:
        if remaining <= 0:
            break
        allocation[item["segment_idx"]] += 1
        remaining -= 1

    while remaining > 0:
        best_choice = None
        for item in eligible:
            current = allocation[item["segment_idx"]]
            if current >= item["capacity"]:
                continue
            score = item["segment_len"] / (current + 1)
            if best_choice is None or score > best_choice[0]:
                best_choice = (score, item["segment_idx"])

        if best_choice is None:
            break

        allocation[best_choice[1]] += 1
        remaining -= 1

    return allocation


def build_segmented_day_splits(
    unique_days,
    n_splits,
    purge_days,
    embargo_days,
    min_train_days,
    min_valid_days,
    gap_days=30,
    min_prefix_days_later_segment=30,
):
    unique_days = pd.Index(sorted(pd.Index(unique_days).unique()))
    segments = split_day_segments(unique_days, gap_days=gap_days)
    allocation = allocate_splits_to_segments(
        segments=segments,
        n_splits=n_splits,
        min_train_days=min_train_days,
        purge_days=purge_days,
        min_valid_days=min_valid_days,
        embargo_days=embargo_days,
        min_prefix_days_later_segment=min_prefix_days_later_segment,
    )

    splits = []
    prior_days = []
    fold_id = 1

    for seg_idx, segment in enumerate(segments):
        n_segment_splits = allocation.get(seg_idx, 0)
        if n_segment_splits == 0:
            prior_days.extend(segment.tolist())
            continue

        start_train_days = min_train_days if seg_idx == 0 else min_prefix_days_later_segment
        available = len(segment) - start_train_days - purge_days - min_valid_days

        if n_segment_splits == 1:
            valid_starts = [start_train_days + purge_days]
        else:
            raw_positions = np.linspace(0, max(available, 0), n_segment_splits)
            valid_starts = [start_train_days + purge_days + int(round(pos)) for pos in raw_positions]

        last_valid_end = -1
        for valid_start in valid_starts:
            valid_start = max(valid_start, last_valid_end + 1 + embargo_days)
            valid_end = min(valid_start + min_valid_days - 1, len(segment) - 1)
            if valid_end <= valid_start:
                continue

            train_end = valid_start - purge_days - 1
            if train_end < 0:
                continue

            train_days = list(prior_days) + list(segment[: train_end + 1])
            valid_days = list(segment[valid_start : valid_end + 1])
            if len(train_days) < min_train_days or len(valid_days) < min_valid_days:
                continue

            embargo_end = min(valid_end + embargo_days, len(segment) - 1)
            splits.append(
                {
                    "fold_id": fold_id,
                    "segment_idx": seg_idx,
                    "train_days": pd.Index(train_days),
                    "valid_days": pd.Index(valid_days),
                    "train_start_day": pd.Timestamp(train_days[0]),
                    "train_end_day": pd.Timestamp(train_days[-1]),
                    "valid_start_day": pd.Timestamp(valid_days[0]),
                    "valid_end_day": pd.Timestamp(valid_days[-1]),
                    "purge_start_day": pd.Timestamp(segment[train_end + 1]) if train_end + 1 < len(segment) else pd.NaT,
                    "purge_end_day": pd.Timestamp(segment[valid_start - 1]) if valid_start - 1 >= 0 else pd.NaT,
                    "embargo_start_day": pd.Timestamp(segment[valid_end + 1]) if valid_end + 1 <= embargo_end else pd.NaT,
                    "embargo_end_day": pd.Timestamp(segment[embargo_end]) if valid_end + 1 <= embargo_end else pd.NaT,
                    "train_day_count": len(train_days),
                    "valid_day_count": len(valid_days),
                }
            )
            last_valid_end = valid_end
            fold_id += 1

        prior_days.extend(segment.tolist())

    if len(splits) == 0:
        raise ValueError("Failed to build any segmented day splits.")
    return splits[:n_splits]


def make_cv_index_pairs(frame, split_records):
    index_pairs = []
    split_meta_rows = []

    for split in split_records:
        train_idx = frame.index[frame[DATE_COL].isin(split["train_days"])].to_numpy()
        valid_idx = frame.index[frame[DATE_COL].isin(split["valid_days"])].to_numpy()
        if len(train_idx) == 0 or len(valid_idx) == 0:
            continue

        index_pairs.append((train_idx, valid_idx))
        split_meta_rows.append(
            {
                "fold_id": split["fold_id"],
                "segment_idx": split["segment_idx"],
                "train_start_day": split["train_start_day"],
                "train_end_day": split["train_end_day"],
                "valid_start_day": split["valid_start_day"],
                "valid_end_day": split["valid_end_day"],
                "purge_start_day": split["purge_start_day"],
                "purge_end_day": split["purge_end_day"],
                "embargo_start_day": split["embargo_start_day"],
                "embargo_end_day": split["embargo_end_day"],
                "train_day_count": split["train_day_count"],
                "valid_day_count": split["valid_day_count"],
                "train_rows": len(train_idx),
                "valid_rows": len(valid_idx),
            }
        )
    return index_pairs, pd.DataFrame(split_meta_rows)


def choose_inner_cv(frame):
    unique_days = pd.Index(sorted(frame[DATE_COL].dropna().unique()))
    try:
        split_records = build_segmented_day_splits(
            unique_days=unique_days,
            n_splits=INNER_N_SPLITS,
            purge_days=INNER_PURGE_DAYS,
            embargo_days=INNER_EMBARGO_DAYS,
            min_train_days=INNER_MIN_TRAIN_DAYS,
            min_valid_days=INNER_MIN_VALID_DAYS,
            gap_days=GAP_SPLIT_DAYS,
            min_prefix_days_later_segment=MIN_PREFIX_DAYS_LATER_SEGMENT,
        )
        cv_pairs, split_df = make_cv_index_pairs(frame, split_records)
        if len(cv_pairs) >= 2:
            return cv_pairs, split_df, "segmented_purged"
    except ValueError:
        pass

    fallback_splits = min(INNER_N_SPLITS, max(frame[DATE_COL].nunique() - 1, 1))
    if fallback_splits < 2:
        raise ValueError("Not enough unique dates to build inner CV.")

    fallback = TimeSeriesSplit(n_splits=fallback_splits)
    fallback_pairs = list(fallback.split(frame))
    fallback_rows = []
    for idx, (train_idx, valid_idx) in enumerate(fallback_pairs, start=1):
        fallback_rows.append(
            {
                "fold_id": idx,
                "segment_idx": -1,
                "train_start_day": frame.iloc[train_idx][DATE_COL].min(),
                "train_end_day": frame.iloc[train_idx][DATE_COL].max(),
                "valid_start_day": frame.iloc[valid_idx][DATE_COL].min(),
                "valid_end_day": frame.iloc[valid_idx][DATE_COL].max(),
                "purge_start_day": pd.NaT,
                "purge_end_day": pd.NaT,
                "embargo_start_day": pd.NaT,
                "embargo_end_day": pd.NaT,
                "train_day_count": frame.iloc[train_idx][DATE_COL].nunique(),
                "valid_day_count": frame.iloc[valid_idx][DATE_COL].nunique(),
                "train_rows": len(train_idx),
                "valid_rows": len(valid_idx),
            }
        )
    return fallback_pairs, pd.DataFrame(fallback_rows), "time_series_fallback"


def build_model_bundle():
    estimator = RandomForestRegressor(
        random_state=RANDOM_STATE,
        n_jobs=1,
    )
    grid = {
        "model__n_estimators": [200, 500],
        "model__max_depth": [3, 5, None],
        "model__min_samples_leaf": [1, 5, 10],
        "model__max_features": ["sqrt", 0.5],
    }
    return estimator, grid


def make_model_pipeline(features):
    estimator, grid = build_model_bundle()
    preprocessor = ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="median")),
                    ]
                ),
                features,
            ),
        ],
        remainder="drop",
    )
    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", estimator),
        ]
    )
    return pipeline, grid


def extract_model_importance(estimator, features):
    fitted_model = estimator.named_steps["model"]
    values = fitted_model.feature_importances_
    return pd.DataFrame(
        {
            "model_name": MODEL_NAME,
            "spec_name": SPEC_NAME,
            "feature_name": features,
            "importance_value": values,
            "abs_importance_value": np.abs(values),
            "importance_type": "feature_importance",
        }
    )


def flatten_summary_columns(frame):
    flat_cols = []
    for col in frame.columns:
        if isinstance(col, tuple):
            flat_cols.append("_".join([str(part) for part in col if part]).strip("_"))
        else:
            flat_cols.append(str(col))
    out = frame.copy()
    out.columns = flat_cols
    return out.reset_index()


def select_two_stage_reference_row(frame, source_label):
    if "variant_key" not in frame.columns:
        raise ValueError(f"Two-stage reference file missing variant_key: {source_label}")

    if source_label == "stage2_all_model_results_table.csv":
        mask = (
            frame["spec_name"].eq("two_stage_v2_core")
            & frame["model_name"].astype(str).str.contains("random_forest", na=False)
        )
        selected = frame.loc[mask].copy()
    else:
        selected = frame.copy()
        if "spec_name" in selected.columns:
            filtered = selected.loc[selected["spec_name"].eq("two_stage_v2_core")].copy()
            if not filtered.empty:
                selected = filtered
        if "rmse_mean" in selected.columns and selected["rmse_mean"].notna().any():
            selected = selected.loc[selected["rmse_mean"].notna()].copy()

    if selected.empty:
        raise ValueError(
            "Could not isolate `RandomForest + two_stage_v2_core` from the two-stage reference."
        )

    if source_label == "stage2_all_model_results_table.csv":
        selected = selected.sort_values(["rmse", "rmse_mean"]).head(1).copy()
    else:
        exact_mask = (
            selected["spec_name"].eq("two_stage_v2_core")
            & selected["model_name"].astype(str).str.contains("random_forest", na=False)
        )
        if exact_mask.any():
            selected = selected.loc[exact_mask].sort_values(["rmse", "rmse_mean"]).head(1).copy()
        else:
            selected = selected.sort_values(["rmse", "rmse_mean"]).head(1).copy()
    return selected.reset_index(drop=True)


def load_two_stage_reference(reference_path=None, holdout_path=None, outer_summary_path=None):
    if reference_path is not None:
        frame = pd.read_csv(reference_path)
        return select_two_stage_reference_row(frame, reference_path.name)

    holdout = pd.read_csv(holdout_path)
    outer = pd.read_csv(outer_summary_path)
    merge_keys = ["variant_key", "model_name", "spec_name"]
    missing_holdout_keys = [key for key in merge_keys if key not in holdout.columns]
    missing_outer_keys = [key for key in merge_keys if key not in outer.columns]
    if missing_holdout_keys or missing_outer_keys:
        raise ValueError(
            "Two-stage holdout/outer references are missing merge keys: "
            f"holdout={missing_holdout_keys}, outer={missing_outer_keys}"
        )

    merged = holdout.merge(
        outer,
        on=merge_keys,
        how="left",
        suffixes=("", "_outer"),
    )
    return select_two_stage_reference_row(merged, f"{holdout_path.name}+{outer_summary_path.name}")


def build_one_stage_reference_frame(holdout_df, outer_summary_df, branch, display_label_prefix, source_run):
    merged = holdout_df.merge(outer_summary_df, on=["model_name", "spec_name"], how="left")
    merged["branch"] = branch
    merged["display_label"] = display_label_prefix + " + " + merged["spec_name"]
    merged["source_run"] = source_run
    merged["variant_key"] = ""
    merged = merged.rename(
        columns={
            "rmse": "holdout_rmse",
            "mae": "holdout_mae",
            "r2": "holdout_r2",
            "sign_accuracy": "holdout_sign_accuracy",
            "rmse_mean": "outer_rmse_mean",
            "rmse_std": "outer_rmse_std",
            "mae_mean": "outer_mae_mean",
            "mae_std": "outer_mae_std",
            "r2_mean": "outer_r2_mean",
            "r2_std": "outer_r2_std",
            "sign_accuracy_mean": "outer_sign_accuracy_mean",
            "sign_accuracy_std": "outer_sign_accuracy_std",
        }
    )
    return merged[
        [
            "branch",
            "display_label",
            "variant_key",
            "model_name",
            "spec_name",
            "source_run",
            "holdout_rmse",
            "holdout_mae",
            "holdout_r2",
            "holdout_sign_accuracy",
            "outer_rmse_mean",
            "outer_rmse_std",
            "outer_mae_mean",
            "outer_mae_std",
            "outer_r2_mean",
            "outer_r2_std",
            "outer_sign_accuracy_mean",
            "outer_sign_accuracy_std",
        ]
    ]


def build_two_stage_reference_frame(two_stage_df):
    frame = two_stage_df.copy()
    frame["branch"] = "two_stage_final"
    frame["display_label"] = "two-stage final"
    frame["source_run"] = "stage2_final_selected_candidate"
    frame = frame.rename(
        columns={
            "rmse": "holdout_rmse",
            "mae": "holdout_mae",
            "r2": "holdout_r2",
            "sign_accuracy": "holdout_sign_accuracy",
            "rmse_mean": "outer_rmse_mean",
            "rmse_std": "outer_rmse_std",
            "mae_mean": "outer_mae_mean",
            "mae_std": "outer_mae_std",
            "r2_mean": "outer_r2_mean",
            "r2_std": "outer_r2_std",
            "sign_accuracy_mean": "outer_sign_accuracy_mean",
            "sign_accuracy_std": "outer_sign_accuracy_std",
        }
    )
    return frame[
        [
            "branch",
            "display_label",
            "variant_key",
            "model_name",
            "spec_name",
            "source_run",
            "holdout_rmse",
            "holdout_mae",
            "holdout_r2",
            "holdout_sign_accuracy",
            "outer_rmse_mean",
            "outer_rmse_std",
            "outer_mae_mean",
            "outer_mae_std",
            "outer_r2_mean",
            "outer_r2_std",
            "outer_sign_accuracy_mean",
            "outer_sign_accuracy_std",
        ]
    ]


def build_benchmark_reference_frame(benchmark_holdout_df, source_run):
    frame = benchmark_holdout_df.copy()
    frame["branch"] = "benchmark_reference"
    frame["display_label"] = frame["benchmark_label"].astype(str)
    frame["model_name"] = frame["reference_model_name"].fillna("benchmark")
    frame["spec_name"] = frame["benchmark_key"].astype(str)
    frame["source_run"] = source_run
    frame = frame.rename(
        columns={
            "rmse": "holdout_rmse",
            "mae": "holdout_mae",
            "r2": "holdout_r2",
            "sign_accuracy": "holdout_sign_accuracy",
        }
    )
    frame["outer_rmse_mean"] = np.nan
    frame["outer_rmse_std"] = np.nan
    frame["outer_mae_mean"] = np.nan
    frame["outer_mae_std"] = np.nan
    frame["outer_r2_mean"] = np.nan
    frame["outer_r2_std"] = np.nan
    frame["outer_sign_accuracy_mean"] = np.nan
    frame["outer_sign_accuracy_std"] = np.nan
    return frame[
        [
            "branch",
            "display_label",
            "variant_key",
            "model_name",
            "spec_name",
            "source_run",
            "holdout_rmse",
            "holdout_mae",
            "holdout_r2",
            "holdout_sign_accuracy",
            "outer_rmse_mean",
            "outer_rmse_std",
            "outer_mae_mean",
            "outer_mae_std",
            "outer_r2_mean",
            "outer_r2_std",
            "outer_sign_accuracy_mean",
            "outer_sign_accuracy_std",
        ]
    ]


def stable_rmse_reading(holdout_delta, outer_delta):
    if holdout_delta < 0 and outer_delta < 0:
        return "Yes, both hold-out RMSE and outer RMSE improve."
    if holdout_delta > 0 and outer_delta > 0:
        return "No, both hold-out RMSE and outer RMSE worsen."
    return "Mixed: hold-out and outer CV do not point in the same direction."


def infer_gap_reading(
    richer_vs_core_holdout_rmse,
    richer_vs_core_outer_rmse,
    richer_vs_macro_holdout_rmse,
    richer_vs_macro_outer_rmse,
    richer_vs_two_holdout_rmse,
    richer_vs_two_outer_rmse,
):
    improves_over_narrow = (
        richer_vs_core_holdout_rmse < 0
        and richer_vs_core_outer_rmse < 0
        and richer_vs_macro_holdout_rmse < 0
        and richer_vs_macro_outer_rmse < 0
    )
    still_behind_two_stage = (
        richer_vs_two_holdout_rmse > 0 and richer_vs_two_outer_rmse > 0
    )

    if improves_over_narrow and still_behind_two_stage:
        return (
            "Richer daily raw representation helps relative to the historical narrow single-stage baselines, "
            "but the remaining gap to two-stage after matching the learner is more consistent with the missing "
            "`P_taco` intermediate supervision / compression layer still adding value."
        )

    if (
        richer_vs_core_holdout_rmse >= 0
        and richer_vs_core_outer_rmse >= 0
        and richer_vs_macro_holdout_rmse >= 0
        and richer_vs_macro_outer_rmse >= 0
    ):
        return (
            "Richer daily raw representation does not reliably outperform the historical narrow baselines, "
            "so the bottleneck still looks closer to direct daily representation / event discrimination quality "
            "than to a pure compression-loss story."
        )

    return (
        "The evidence is mixed: richer representation changes the single-stage frontier, but not cleanly enough "
        "to separate intermediate-supervision value from residual event-level discrimination differences."
    )

In [ ]:
required_reference_paths = [DATA_PATH]
if TWO_STAGE_REFERENCE_PATH is not None:
    required_reference_paths.append(TWO_STAGE_REFERENCE_PATH)
else:
    required_reference_paths.extend(
        [TWO_STAGE_REFERENCE_HOLDOUT_PATH, TWO_STAGE_REFERENCE_OUTER_SUMMARY_PATH]
    )
missing_reference_paths = [str(path.relative_to(PROJECT_ROOT)) for path in required_reference_paths if not path.exists()]
if missing_reference_paths:
    raise FileNotFoundError(f"Missing required input files: {missing_reference_paths}")

df = pd.read_csv(DATA_PATH)
df[DATE_COL] = pd.to_datetime(df[DATE_COL]).dt.normalize()
df = df.sort_values(DATE_COL).reset_index(drop=True)

required_columns = sorted(set([TARGET_COL, DATE_COL, "stage2_split"] + MODEL_FEATURES))
missing_required = [col for col in required_columns if col not in df.columns]
if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

forbidden_cols = [
    col
    for col in [
        "label_retreat",
        "p_taco",
        "follow_up_count_all_48h",
        "follow_up_count_relevant_48h",
    ]
    if col in df.columns
]
if forbidden_cols:
    raise ValueError(f"Forbidden single-stage input columns present: {forbidden_cols}")

main_df = df[df["stage2_split"].isin([TRAIN_LABEL, HOLDOUT_LABEL])].copy()
main_df = main_df.dropna(subset=[TARGET_COL]).sort_values(DATE_COL).reset_index(drop=True)

split_summary = (
    main_df.groupby("stage2_split")
    .agg(
        n_rows=(DATE_COL, "size"),
        n_days=(DATE_COL, "nunique"),
        min_date=(DATE_COL, "min"),
        max_date=(DATE_COL, "max"),
    )
    .reset_index()
)
display(split_summary)

missing_summary = pd.DataFrame(
    {
        "feature_name": MODEL_FEATURES,
        "missing_rate": [float(main_df[col].isna().mean()) for col in MODEL_FEATURES],
    }
).sort_values(["missing_rate", "feature_name"], ascending=[False, True])
display(missing_summary.head(20))

two_stage_df = load_two_stage_reference(
    reference_path=TWO_STAGE_REFERENCE_PATH,
    holdout_path=TWO_STAGE_REFERENCE_HOLDOUT_PATH,
    outer_summary_path=TWO_STAGE_REFERENCE_OUTER_SUMMARY_PATH,
)
benchmark_holdout_df = pd.read_csv(BENCHMARK_HOLDOUT_PATH) if BENCHMARK_HOLDOUT_PATH is not None else None
display(two_stage_df)

In [5]:
outer_unique_days = pd.Index(
    sorted(main_df.loc[main_df["stage2_split"] == TRAIN_LABEL, DATE_COL].dropna().unique())
)
outer_split_records = build_segmented_day_splits(
    unique_days=outer_unique_days,
    n_splits=OUTER_N_SPLITS,
    purge_days=OUTER_PURGE_DAYS,
    embargo_days=OUTER_EMBARGO_DAYS,
    min_train_days=OUTER_MIN_TRAIN_DAYS,
    min_valid_days=OUTER_MIN_VALID_DAYS,
    gap_days=GAP_SPLIT_DAYS,
    min_prefix_days_later_segment=MIN_PREFIX_DAYS_LATER_SEGMENT,
)
_, outer_splits_df = make_cv_index_pairs(
    frame=main_df[main_df["stage2_split"] == TRAIN_LABEL].copy(),
    split_records=outer_split_records,
)
display(outer_splits_df)

train_pool_df = main_df[main_df["stage2_split"] == TRAIN_LABEL].copy().reset_index(drop=True)
holdout_df = main_df[main_df["stage2_split"] == HOLDOUT_LABEL].copy().reset_index(drop=True)

outer_fold_metric_rows = []
outer_prediction_frames = []
best_param_rows = []

for outer_split in outer_split_records:
    outer_train = train_pool_df[train_pool_df[DATE_COL].isin(outer_split["train_days"])].copy().reset_index(drop=True)
    outer_valid = train_pool_df[train_pool_df[DATE_COL].isin(outer_split["valid_days"])].copy().reset_index(drop=True)

    inner_cv_pairs, inner_splits_df, inner_strategy = choose_inner_cv(outer_train)
    pipeline, param_grid = make_model_pipeline(MODEL_FEATURES)
    search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        scoring="neg_root_mean_squared_error",
        cv=inner_cv_pairs,
        refit=True,
        n_jobs=None,
    )
    search.fit(outer_train[MODEL_FEATURES], outer_train[TARGET_COL])

    valid_pred = search.best_estimator_.predict(outer_valid[MODEL_FEATURES])
    fold_metrics = compute_regression_metrics(
        outer_valid[TARGET_COL].to_numpy(),
        valid_pred,
    )
    fold_metrics.update(
        {
            "model_name": MODEL_NAME,
            "spec_name": SPEC_NAME,
            "fold_id": outer_split["fold_id"],
            "segment_idx": outer_split["segment_idx"],
            "inner_cv_strategy": inner_strategy,
            "best_params_json": json.dumps(search.best_params_, ensure_ascii=False),
            "best_inner_rmse": float(-search.best_score_),
            "train_start_day": outer_split["train_start_day"],
            "train_end_day": outer_split["train_end_day"],
            "valid_start_day": outer_split["valid_start_day"],
            "valid_end_day": outer_split["valid_end_day"],
            "train_day_count": outer_split["train_day_count"],
            "valid_day_count": outer_split["valid_day_count"],
            "train_rows": len(outer_train),
            "valid_rows": len(outer_valid),
        }
    )
    outer_fold_metric_rows.append(fold_metrics)

    outer_prediction_frames.append(
        pd.DataFrame(
            {
                "date": outer_valid[DATE_COL].to_numpy(),
                "model_name": MODEL_NAME,
                "spec_name": SPEC_NAME,
                "fold_id": outer_split["fold_id"],
                "y_true": outer_valid[TARGET_COL].to_numpy(),
                "y_pred": valid_pred,
            }
        )
    )

    best_param_rows.append(
        {
            "model_name": MODEL_NAME,
            "spec_name": SPEC_NAME,
            "fold_id": outer_split["fold_id"],
            "segment_idx": outer_split["segment_idx"],
            "best_params_json": json.dumps(search.best_params_, ensure_ascii=False),
            "best_inner_rmse": float(-search.best_score_),
            "inner_cv_strategy": inner_strategy,
            "feature_list_json": json.dumps(MODEL_FEATURES, ensure_ascii=False),
        }
    )

full_inner_pairs, full_inner_splits_df, full_inner_strategy = choose_inner_cv(train_pool_df)
full_pipeline, full_param_grid = make_model_pipeline(MODEL_FEATURES)
full_search = GridSearchCV(
    estimator=full_pipeline,
    param_grid=full_param_grid,
    scoring="neg_root_mean_squared_error",
    cv=full_inner_pairs,
    refit=True,
    n_jobs=None,
)
full_search.fit(train_pool_df[MODEL_FEATURES], train_pool_df[TARGET_COL])

holdout_pred = full_search.best_estimator_.predict(holdout_df[MODEL_FEATURES])
holdout_metrics = compute_regression_metrics(
    holdout_df[TARGET_COL].to_numpy(),
    holdout_pred,
)
holdout_metrics.update(
    {
        "model_name": MODEL_NAME,
        "spec_name": SPEC_NAME,
        "best_params_json": json.dumps(full_search.best_params_, ensure_ascii=False),
        "best_inner_rmse": float(-full_search.best_score_),
        "inner_cv_strategy": full_inner_strategy,
        "n_obs": len(holdout_df),
    }
)

outer_fold_metrics_df = pd.DataFrame(outer_fold_metric_rows)
outer_predictions_df = pd.concat(outer_prediction_frames, ignore_index=True)
best_params_df = pd.DataFrame(best_param_rows)
holdout_metrics_df = pd.DataFrame([holdout_metrics])
holdout_predictions_df = pd.DataFrame(
    {
        "date": holdout_df[DATE_COL].to_numpy(),
        "model_name": MODEL_NAME,
        "spec_name": SPEC_NAME,
        "y_true": holdout_df[TARGET_COL].to_numpy(),
        "y_pred": holdout_pred,
    }
)
holdout_importances_df = extract_model_importance(
    estimator=full_search.best_estimator_,
    features=MODEL_FEATURES,
)

outer_summary_df = flatten_summary_columns(
    outer_fold_metrics_df.groupby(["model_name", "spec_name"]).agg(
        {
            "rmse": ["mean", "std"],
            "mae": ["mean", "std"],
            "r2": ["mean", "std"],
            "sign_accuracy": ["mean", "std"],
        }
    )
)

display(outer_fold_metrics_df)
display(outer_summary_df)
display(holdout_metrics_df)
display(
    holdout_importances_df.sort_values("abs_importance_value", ascending=False).head(20)
)

,fold_id,segment_idx,train_start_day,train_end_day,valid_start_day,valid_end_day,purge_start_day,purge_end_day,embargo_start_day,embargo_end_day,train_day_count,valid_day_count,train_rows,valid_rows
0,1,0,2017-01-20,2017-10-04,2017-10-09,2017-11-17,2017-10-05,2017-10-06,2017-11-20,2017-11-27,180,30,180,30
1,2,0,2017-01-20,2018-10-19,2018-10-24,2018-12-06,2018-10-22,2018-10-23,2018-12-07,2018-12-13,444,30,444,30
2,3,0,2017-01-20,2019-11-06,2019-11-11,2019-12-23,2019-11-07,2019-11-08,2019-12-24,2019-12-31,708,30,708,30
3,4,0,2017-01-20,2020-11-20,2020-11-25,2021-01-08,2020-11-23,2020-11-24,NaT,NaT,972,30,972,30
4,5,1,2017-01-20,2025-02-28,2025-03-05,2025-04-15,2025-03-03,2025-03-04,2025-04-16,2025-04-22,1034,30,1034,30


,rmse,mae,r2,sign_accuracy,actual_mean,pred_mean,model_name,spec_name,fold_id,segment_idx,...,best_params_json,best_inner_rmse,train_start_day,train_end_day,valid_start_day,valid_end_day,train_day_count,valid_day_count,train_rows,valid_rows
0,0.005393,0.004521,-0.024626,0.500000,0.000460,0.000317,random_forest_regressor,single_stage_daily_raw_rich,1,0,...,"{""model__max_depth"": null, ""model__max_feature...",0.006433,2017-01-20,2017-10-04,2017-10-09,2017-11-17,180,30,180,30
1,0.005319,0.004154,-0.090590,0.566667,0.000181,-0.000379,random_forest_regressor,single_stage_daily_raw_rich,2,0,...,"{""model__max_depth"": null, ""model__max_feature...",0.006481,2017-01-20,2018-10-19,2018-10-24,2018-12-06,444,30,444,30
2,0.004489,0.003395,-0.086356,0.533333,0.000619,0.000334,random_forest_regressor,single_stage_daily_raw_rich,3,0,...,"{""model__max_depth"": null, ""model__max_feature...",0.005544,2017-01-20,2019-11-06,2019-11-11,2019-12-23,708,30,708,30
3,0.011253,0.007860,-0.007797,0.633333,0.000735,0.000281,random_forest_regressor,single_stage_daily_raw_rich,4,0,...,"{""model__max_depth"": 5, ""model__max_features"":...",0.007990,2017-01-20,2020-11-20,2020-11-25,2021-01-08,972,30,972,30
4,0.013067,0.009350,-0.007933,0.700000,0.003500,-0.000318,random_forest_regressor,single_stage_daily_raw_rich,5,1,...,"{""model__max_depth"": 5, ""model__max_features"":...",0.007187,2017-01-20,2025-02-28,2025-03-05,2025-04-15,1034,30,1034,30


,model_name,spec_name,rmse_mean,rmse_std,mae_mean,mae_std,r2_mean,r2_std,sign_accuracy_mean,sign_accuracy_std
0,random_forest_regressor,single_stage_daily_raw_rich,0.007904,0.003954,0.005856,0.002596,-0.04346,0.041683,0.586667,0.080277


,rmse,mae,r2,sign_accuracy,actual_mean,pred_mean,model_name,spec_name,best_params_json,best_inner_rmse,inner_cv_strategy,n_obs
0,0.01453,0.009345,-0.02635,0.621622,0.002761,0.000604,random_forest_regressor,single_stage_daily_raw_rich,"{""model__max_depth"": 5, ""model__max_features"":...",0.008244,segmented_purged,74


,model_name,spec_name,feature_name,importance_value,abs_importance_value,importance_type
3,random_forest_regressor,single_stage_daily_raw_rich,vix_change_5d_lag1,0.128849,0.128849,feature_importance
2,random_forest_regressor,single_stage_daily_raw_rich,vix_ma_5d_lag1,0.118120,0.118120,feature_importance
5,random_forest_regressor,single_stage_daily_raw_rich,spx_drawdown_5d_lag1,0.098504,0.098504,feature_importance
4,random_forest_regressor,single_stage_daily_raw_rich,spx_ret_1d_lag1,0.096867,0.096867,feature_importance
0,random_forest_regressor,single_stage_daily_raw_rich,dxy_ret_1d_lag1,0.069377,0.069377,feature_importance
1,random_forest_regressor,single_stage_daily_raw_rich,real_yield_change_5d_lag1,0.050169,0.050169,feature_importance
37,random_forest_regressor,single_stage_daily_raw_rich,trend_war_z_60d_lag1,0.049794,0.049794,feature_importance
35,random_forest_regressor,single_stage_daily_raw_rich,trend_tariff_z_60d_lag1,0.034616,0.034616,feature_importance
36,random_forest_regressor,single_stage_daily_raw_rich,trend_inflation_z_60d_lag1,0.031474,0.031474,feature_importance
14,random_forest_regressor,single_stage_daily_raw_rich,finbert_neg_top2_sum,0.024936,0.024936,feature_importance


In [6]:
outer_splits_df.to_csv(RESULTS_DIR / "outer_splits.csv", index=False)
best_params_df.to_csv(RESULTS_DIR / "best_params_by_fold.csv", index=False)
outer_fold_metrics_df.to_csv(RESULTS_DIR / "outer_fold_metrics.csv", index=False)
outer_summary_df.to_csv(RESULTS_DIR / "outer_summary_metrics.csv", index=False)
outer_predictions_df.to_csv(RESULTS_DIR / "outer_predictions.csv", index=False)
holdout_metrics_df.to_csv(RESULTS_DIR / "holdout_metrics.csv", index=False)
holdout_predictions_df.to_csv(RESULTS_DIR / "holdout_predictions.csv", index=False)
holdout_importances_df.to_csv(RESULTS_DIR / "holdout_importances.csv", index=False)

print("Exported protocol outputs to:", RESULTS_DIR)

Exported protocol outputs to: /Users/horace/Coding/ML finance/project/main_rerun/artifacts/ablation/one_stage_daily_raw_rich_modeling


In [ ]:
richer_reference_df = build_one_stage_reference_frame(
    holdout_df=holdout_metrics_df,
    outer_summary_df=outer_summary_df,
    branch="single_stage_formal_rf",
    display_label_prefix="formal single-stage RandomForest",
    source_run="one_stage_daily_raw_rich_modeling_protocol_v1",
)
two_stage_reference_df = build_two_stage_reference_frame(two_stage_df)
richer_holdout_row = holdout_metrics_df.iloc[0]
richer_outer_row = outer_summary_df.iloc[0]
two_stage_row = two_stage_df.iloc[0]

reference_frames = [two_stage_reference_df, richer_reference_df]
if benchmark_holdout_df is not None:
    benchmark_reference_df = build_benchmark_reference_frame(
        benchmark_holdout_df=benchmark_holdout_df,
        source_run="main_rerun_stage2_final_compare",
    )
    reference_frames.append(benchmark_reference_df)

refreshed_table_df = pd.concat(reference_frames, ignore_index=True)
refreshed_table_df["sort_key"] = refreshed_table_df["display_label"].map(
    {
        "two-stage final": 1,
        "formal single-stage RandomForest + single_stage_daily_raw_rich": 2,
        "RandomForest + macro_only": 3,
        "Ridge + macro_only": 4,
        "Random Walk (Zero)": 5,
    }
).fillna(999)
refreshed_table_df = (
    refreshed_table_df.sort_values(["sort_key", "display_label"])
    .drop(columns=["sort_key"])
    .reset_index(drop=True)
)
refreshed_table_df.to_csv(RESULTS_DIR / "refreshed_one_stage_vs_two_stage_table.csv", index=False)

richer_vs_two_holdout_rmse = float(richer_holdout_row["rmse"] - two_stage_row["rmse"])
richer_vs_two_outer_rmse = float(richer_outer_row["rmse_mean"] - two_stage_row["rmse_mean"])
richer_vs_two_holdout_mae = float(richer_holdout_row["mae"] - two_stage_row["mae"])
richer_vs_two_holdout_r2 = float(richer_holdout_row["r2"] - two_stage_row["r2"])
richer_vs_two_holdout_sign = float(richer_holdout_row["sign_accuracy"] - two_stage_row["sign_accuracy"])

summary_lines = [
    "# Refreshed One-Stage vs Two-Stage Summary",
    "",
    "## Formal Compare Setup",
    f"- New formal single-stage spec: `{SPEC_NAME}`",
    f"- Learner: `{MODEL_NAME}`",
    f"- Holdout window label: `{HOLDOUT_LABEL}`",
    f"- Two-stage anchor: `{two_stage_row['variant_key']} + {two_stage_row['model_name']} + {two_stage_row['spec_name']}`",
    (
        f"- Two-stage reference file: `{TWO_STAGE_REFERENCE_PATH.relative_to(PROJECT_ROOT)}`"
        if TWO_STAGE_REFERENCE_PATH is not None
        else
        f"- Two-stage reference files: `{TWO_STAGE_REFERENCE_HOLDOUT_PATH.relative_to(PROJECT_ROOT)}` + "
        f"`{TWO_STAGE_REFERENCE_OUTER_SUMMARY_PATH.relative_to(PROJECT_ROOT)}`"
    ),
]

if benchmark_holdout_df is not None:
    benchmark_by_key = benchmark_holdout_df.set_index("benchmark_key")
    macro_rf_row = benchmark_by_key.loc["macro_only_random_forest"]
    ridge_row = benchmark_by_key.loc["macro_only_ridge"]
    random_walk_row = benchmark_by_key.loc["random_walk_zero"]
    richer_vs_macro_holdout_rmse = float(richer_holdout_row["rmse"] - macro_rf_row["rmse"])
    summary_lines.extend(
        [
            f"- Rerun benchmark file: `{BENCHMARK_HOLDOUT_PATH.relative_to(PROJECT_ROOT)}`",
            "",
            "## Q1: Does `single_stage_daily_raw_rich` add value over `RandomForest + macro_only`?",
            f"- Hold-out RMSE delta (`rich - RF macro_only`): `{richer_vs_macro_holdout_rmse:.6f}`",
            "- Outer delta versus `RF macro_only` is unavailable in this rerun because `macro_only` was kept only as a benchmark row, not as an outer-CV selection candidate.",
            "",
        ]
    )
else:
    summary_lines.extend(
        [
            "- Rerun benchmark file: unavailable in this run.",
            "",
        ]
    )

summary_lines.extend(
    [
        "## Q2: How far is richer single-stage from current two-stage final?",
        f"- Hold-out RMSE delta (`rich - two-stage final`): `{richer_vs_two_holdout_rmse:.6f}`",
        f"- Outer RMSE delta (`rich - two-stage final`): `{richer_vs_two_outer_rmse:.6f}`",
        f"- Hold-out MAE delta (`rich - two-stage final`): `{richer_vs_two_holdout_mae:.6f}`",
        f"- Hold-out R2 delta (`rich - two-stage final`): `{richer_vs_two_holdout_r2:.6f}`",
        f"- Hold-out sign accuracy delta (`rich - two-stage final`): `{richer_vs_two_holdout_sign:.6f}`",
    ]
)

if benchmark_holdout_df is not None:
    summary_lines.extend(
        [
            "",
            "## Benchmark Context",
            f"- RF macro_only hold-out RMSE: `{macro_rf_row['rmse']:.6f}`",
            f"- Ridge macro_only hold-out RMSE: `{ridge_row['rmse']:.6f}`",
            f"- Random walk hold-out RMSE: `{random_walk_row['rmse']:.6f}`",
        ]
    )

summary_path = RESULTS_DIR / "refreshed_one_stage_vs_two_stage_summary.md"
summary_path.write_text("\n".join(summary_lines) + "\n", encoding="utf-8")

display(refreshed_table_df)
display(Markdown(summary_path.read_text(encoding="utf-8")))
print(summary_path)